# 06 — Model Export for Deployment

Final step: save the trained models and everything a deployment layer (a dashboard or an API) needs to make predictions **without retraining**.

Training and inference are kept separate: a notebook trains and saves files; the app only **loads** them. Each saved file contains the model together with its exact feature list, so the app cannot feed the wrong columns.

Files saved:
- `energy_forecast_1h.pkl` — 1-hour-ahead delta Random Forest (+ feature list + base lag).
- `energy_forecast_24h.pkl` — day-ahead delta Random Forest.
- `anomaly_model.pkl` — plain Random Forest + the **validation-calibrated** anomaly numbers (`median`, `sigma`, `eps`, default `K`).
- `deployment_config.json` — human-readable settings.

## 6.1 Load the trained models (from notebook 02)

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

IN_DIR1 = "/kaggle/input/notebooks/azizullahasad/01-data-cleaning-and-eda"
IN_DIR = "/kaggle/input/notebooks/azizullahasad/02-feature-enginering-and-baseline-model"    
WORK_DIR = "/kaggle/working/"

b_1h = joblib.load(f"{IN_DIR}/rf_delta_1h.pkl")
b_24h = joblib.load(f"{IN_DIR}/rf_delta_24h.pkl")
b_plain = joblib.load(f"{IN_DIR}/rf_plain_1h.pkl")

hourly = pd.read_csv(f"{IN_DIR1}/house38_hourly_clean.csv", index_col=0, parse_dates=True)
val = pd.read_csv(f"{IN_DIR}/house38_val.csv", index_col=0, parse_dates=True)
test = pd.read_csv(f"{IN_DIR}/house38_test.csv", index_col=0, parse_dates=True)

## 6.2 Anomaly statistics from the validation set

The anomaly threshold must come from data the model has **not** been scored on, so we compute `median` and `sigma` of the relative residual on the **validation** period (same method as notebook 04).

In [2]:
rf_plain = b_plain["model"]
features = b_plain["features"]

pred_val = rf_plain.predict(val[features])
eps = 0.1 * val["y"].mean()
rel_val = (val["y"].values - pred_val) / (pred_val + eps)
median = float(np.median(rel_val))
sigma = float(1.4826 * np.median(np.abs(rel_val - median)))
print("median:", round(median, 4), "| sigma:", round(sigma, 4), "| eps:", round(float(eps), 4))

median: 0.1227 | sigma: 0.4095 | eps: 0.1639


## 6.3 Save the deployment files

In [3]:
joblib.dump(b_1h, f"{WORK_DIR}/energy_forecast_1h.pkl", compress=3)
joblib.dump(b_24h, f"{WORK_DIR}/energy_forecast_24h.pkl", compress=3)
joblib.dump({"model": rf_plain, "features": features, "eps": float(eps), "median": median,
             "sigma": sigma, "k_default": 5}, f"{WORK_DIR}/anomaly_model.pkl", compress=3)

config = {"target": "Usage (kW)", "frequency": "hourly",
          "forecast_1h": {"features": b_1h["features"], "base": b_1h["base"]},
          "forecast_24h": {"features": b_24h["features"], "base": b_24h["base"]},
          "anomaly": {"features": features, "k_default": 5, "k_range": [3.5, 8],
                      "median": median, "sigma": sigma, "eps": float(eps)}}
with open(f"{WORK_DIR}/deployment_config.json", "w") as f:
    json.dump(config, f, indent=2)

import os
for name in ["energy_forecast_1h.pkl", "energy_forecast_24h.pkl", "anomaly_model.pkl", "deployment_config.json"]:
    print(f"{name:28s} {os.path.getsize(f'{WORK_DIR}/{name}') / 1e6:.2f} MB")

energy_forecast_1h.pkl       16.61 MB
energy_forecast_24h.pkl      16.35 MB
anomaly_model.pkl            16.51 MB
deployment_config.json       0.00 MB


## 6.4 Prove that the saved model works from raw history

A deployed app does not have our feature table. It only has the recent hourly usage history. The function below builds the features for the **next hour** from a history series, exactly like training did, and predicts. We then check that it gives the same answer as the training pipeline for a real test hour.

In [4]:
def features_for_next_hour(history, next_time):
    # history: hourly usage values up to next_time - 1 hour (at least 168 values, no gaps)
    s = history.values
    return pd.DataFrame([{
        "hour": next_time.hour,
        "dow": next_time.dayofweek,
        "is_weekend": int(next_time.dayofweek >= 5),
        "lag_1": s[-1], "lag_2": s[-2], "lag_24": s[-24], "lag_48": s[-48], "lag_168": s[-168],
        "roll_mean_24": s[-24:].mean(),
        "roll_std_24": s[-24:].std(ddof=1),
        "roll_mean_168": s[-168:].mean(),
    }])


def predict_next_hour(history, next_time, bundle):
    X = features_for_next_hour(history, next_time)[bundle["features"]]
    return float(X[bundle["base"]].iloc[0] + bundle["model"].predict(X)[0])


# reload from disk, as the app would
loaded_1h = joblib.load(f"{WORK_DIR}/energy_forecast_1h.pkl")

t = test.index[100]
history = hourly["Usage (kW)"].loc[: t - pd.Timedelta(hours=1)]

from_history = predict_next_hour(history, t, loaded_1h)
from_pipeline = float(test.loc[t, "lag_1"] + loaded_1h["model"].predict(test.loc[[t], loaded_1h["features"]])[0])

print("Time           :", t)
print("From history   :", round(from_history, 4), "kW")
print("From pipeline  :", round(from_pipeline, 4), "kW")
print("Actual         :", round(float(test.loc[t, "y"]), 4), "kW")
print("Same answer    :", np.isclose(from_history, from_pipeline))

Time           : 2024-06-13 20:00:00
From history   : 0.968 kW
From pipeline  : 0.968 kW
Actual         : 0.4332 kW
Same answer    : True


## 6.5 Next step (outside this notebook)

Download these files from the Kaggle **Output** panel, put them in your project folder next to the app, and **only load** them in the app — no training code runs in deployment. Each file already contains its feature list, so the app should use `bundle["features"]` instead of hard-coding column names.

Model files of about 15 MB each are fine for GitHub (limit 100 MB per file). Saving a separate model for all 18 houses would be several hundred MB, so a showcase model for one house is enough, while the other houses can be shown from their saved predictions.